In [2]:
import json
from kafka import KafkaProducer

BOOTSTRAP_SERVERS = "localhost:9092,localhost:9094,localhost:9095"
TOPIC = "flights"

_producer = None

FLIGHT_SCHEMA = {
    "type": "struct",
    "fields": [
        {"field": "flight_id", "type": "string", "optional": False},
        {"field": "fleet", "type": "string", "optional": False},
        {"field": "great_circle_distance_nm", "type": "double", "optional": False},
        {"field": "gross_weight_at_liftoff_kg", "type": "double", "optional": False},
        {"field": "max_tailwind_during_takeoff_kt", "type": "double", "optional": False},
        {"field": "generated_at", "type": "string", "optional": False},
    ],
    "optional": False,
    "name": "flight",
}


def get_producer():
    global _producer
    if _producer is None:
        _producer = KafkaProducer(
            bootstrap_servers=BOOTSTRAP_SERVERS,
            key_serializer=lambda k: k.encode("utf-8"),
            value_serializer=lambda v: json.dumps(v).encode("utf-8"),
            acks="all",
            retries=3,
        )
    return _producer


def publish_flight(flight_dict):
    producer = get_producer()
    envelope = {"schema": FLIGHT_SCHEMA, "payload": flight_dict}
    key = flight_dict["fleet"]   # <- partition key: same fleet, same partition, every time
    producer.send(TOPIC, key=key, value=envelope)
    producer.flush()